# CerviGrade — Phase 2: Model training

Before running anything: attach SIPaKMeD, Mendeley LBC, and `cervigrade-cric-dataset` as Input (same three as Phase 1), and this time turn the **Accelerator to GPU** in notebook settings, since actual training happens here. Internet can stay off, nothing needs downloading in this notebook.

This notebook starts by reproducing the exact same manifests and train/val/test split built in the Phase 1 acquisition notebook (same code, same random seed, so it's guaranteed to match), then moves into training EfficientNetB0 on SIPaKMeD, the first of the eight planned runs.

## Locate the three datasets

In [1]:
import os

def find_containing(root, predicate, max_matches=5):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if predicate(dirpath, dirnames, filenames):
            matches.append(dirpath)
            if len(matches) >= max_matches:
                break
    return matches

INPUT_ROOT = "/kaggle/input"

sipakmed_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: "CROPPED" in dn)
print("SIPaKMeD CROPPED folder candidates:")
for m in sipakmed_matches:
    print(" ", os.path.join(m, "CROPPED"))

mendeley_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: "Negative for Intraepithelial malignancy" in dn)
print("\nMendeley root candidates:", mendeley_matches)
mendeley_root = mendeley_matches[0]

cric_cropped_matches = []
for root in ["/kaggle/working", "/kaggle/input"]:
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "CRIC_cropped":
            cric_cropped_matches.append(dirpath)
print("\nCRIC_cropped candidates:", cric_cropped_matches)
CRIC_CROPPED_DIR = cric_cropped_matches[0]

SIPaKMeD CROPPED folder candidates:
  /kaggle/input/datasets/marinaeplissiti/sipakmed/im_Parabasal/im_Parabasal/CROPPED
  /kaggle/input/datasets/marinaeplissiti/sipakmed/im_Dyskeratotic/im_Dyskeratotic/CROPPED
  /kaggle/input/datasets/marinaeplissiti/sipakmed/im_Metaplastic/im_Metaplastic/CROPPED
  /kaggle/input/datasets/marinaeplissiti/sipakmed/im_Superficial-Intermediate/im_Superficial-Intermediate/CROPPED
  /kaggle/input/datasets/marinaeplissiti/sipakmed/im_Koilocytotic/im_Koilocytotic/CROPPED

Mendeley root candidates: ['/kaggle/input/datasets/blank1508/mendeley-lbc-cervical-cancer']

CRIC_cropped candidates: ['/kaggle/input/datasets/wagnermushayija/cervigrade-cric-dataset/CRIC_cropped']


## Rebuild the manifests and reproduce Phase 1's train/val/test split

Same grouping-by-source-image logic and the same fixed seed as the acquisition notebook, so this reproduces the identical split without needing to carry CSV files between notebooks.

In [2]:
import re
import random
import pandas as pd

RANDOM_SEED = 42

def build_cric_manifest():
    rows = []
    for cls in sorted(os.listdir(CRIC_CROPPED_DIR)):
        cls_dir = os.path.join(CRIC_CROPPED_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            slide_id = fname.split("_")[0]
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "CRIC", "group": f"CRIC_slide_{slide_id}"})
    return pd.DataFrame(rows)

def build_sipakmed_manifest():
    rows = []
    for m in sipakmed_matches:
        cls = os.path.basename(m)
        cropped_dir = os.path.join(m, "CROPPED")
        for fname in os.listdir(cropped_dir):
            if not fname.lower().endswith(".bmp"):
                continue
            field_id = fname.split("_")[0]
            rows.append({"filepath": os.path.join(cropped_dir, fname), "label": cls,
                         "dataset": "SIPaKMeD", "group": f"SIPaKMeD_{cls}_{field_id}"})
    return pd.DataFrame(rows)

def build_mendeley_manifest():
    rows = []
    for cls in sorted(os.listdir(mendeley_root)):
        cls_dir = os.path.join(mendeley_root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            photo_id = re.split(r"\s*\(", fname)[0].strip()
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "Mendeley", "group": f"Mendeley_{photo_id}"})
    return pd.DataFrame(rows)

cric_manifest = build_cric_manifest()
sipakmed_manifest = build_sipakmed_manifest()
mendeley_manifest = build_mendeley_manifest()

def allocate_group_counts(n, train_frac=0.7, val_frac=0.15, test_frac=0.15):
    if n == 1:
        return 1, 0, 0
    if n == 2:
        return 1, 0, 1
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac)) if n - n_test >= 2 else 0
    n_train = n - n_test - n_val
    if n_train < 1:
        n_train = 1
        if n_val > 0:
            n_val -= 1
        else:
            n_test -= 1
    return n_train, n_val, n_test

def split_by_group_per_class(manifest_df, seed=RANDOM_SEED):
    rng = random.Random(seed)
    group_split = {}
    for label, sub in manifest_df.groupby("label"):
        groups = sorted(sub["group"].unique())
        rng.shuffle(groups)
        n_train, n_val, n_test = allocate_group_counts(len(groups))
        for g in groups[:n_train]:
            group_split[g] = "train"
        for g in groups[n_train:n_train + n_val]:
            group_split[g] = "val"
        for g in groups[n_train + n_val:n_train + n_val + n_test]:
            group_split[g] = "test"
    return manifest_df["group"].map(group_split)

def split_by_group_global(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(manifest_df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train = round(n * train_frac)
    n_val = round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]:
        group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]:
        group_split[g] = "val"
    for g in groups[n_train + n_val:]:
        group_split[g] = "test"
    return manifest_df["group"].map(group_split)

cric_manifest["split"] = split_by_group_global(cric_manifest)
sipakmed_manifest["split"] = split_by_group_per_class(sipakmed_manifest)
mendeley_manifest["split"] = split_by_group_per_class(mendeley_manifest)

print("Manifest sizes:", len(cric_manifest), len(sipakmed_manifest), len(mendeley_manifest))
print(sipakmed_manifest.groupby(["label", "split"]).size().unstack(fill_value=0))

Manifest sizes: 11534 4049 962
split                        test  train  val
label                                        
im_Dyskeratotic               139    566  108
im_Koilocytotic               138    570  117
im_Metaplastic                110    551  132
im_Parabasal                   98    558  131
im_Superficial-Intermediate   123    598  110


## Set up EfficientNetB0

EfficientNetB0 expects 224x224 inputs. This sets that up along with the class list for SIPaKMeD's 5 cell types.

In [3]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, models

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

tf.random.set_seed(RANDOM_SEED)

sipakmed_classes = sorted(sipakmed_manifest["label"].unique())
class_to_idx = {c: i for i, c in enumerate(sipakmed_classes)}
print("SIPaKMeD classes:", class_to_idx)

SIPaKMeD classes: {'im_Dyskeratotic': 0, 'im_Koilocytotic': 1, 'im_Metaplastic': 2, 'im_Parabasal': 3, 'im_Superficial-Intermediate': 4}


## Build the data pipeline for SIPaKMeD

Reads each image straight from its file path, resizes it, applies light augmentation (flips, brightness) only on the training set, and batches everything for the GPU.

In [4]:
def make_dataset(df, class_to_idx, img_size, batch_size, shuffle=False, augment=False):
    filepaths = df["filepath"].values
    labels = df["label"].map(class_to_idx).values

    def load_and_preprocess(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [img_size, img_size])
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            image = tf.image.random_brightness(image, 0.1)
        image = preprocess_input(image)
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

sipakmed_train_df = sipakmed_manifest[sipakmed_manifest["split"] == "train"]
sipakmed_val_df = sipakmed_manifest[sipakmed_manifest["split"] == "val"]
sipakmed_test_df = sipakmed_manifest[sipakmed_manifest["split"] == "test"]

train_ds = make_dataset(sipakmed_train_df, class_to_idx, IMG_SIZE, BATCH_SIZE, shuffle=True, augment=True)
val_ds = make_dataset(sipakmed_val_df, class_to_idx, IMG_SIZE, BATCH_SIZE)
test_ds = make_dataset(sipakmed_test_df, class_to_idx, IMG_SIZE, BATCH_SIZE)

print(f"Train: {len(sipakmed_train_df)}  Val: {len(sipakmed_val_df)}  Test: {len(sipakmed_test_df)}")

I0000 00:00:1790459708.405374      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1790459708.408220      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Train: 2843  Val: 598  Test: 608


## Build EfficientNetB0 with a new head for SIPaKMeD's 5 classes

Starts from ImageNet weights with the base frozen, and adds a small new classification head on top. Freezing the base first is standard practice, it trains fast and avoids wrecking the pretrained features before the new head has learned anything useful.

In [5]:
def build_efficientnetb0(num_classes, img_size):
    base_model = EfficientNetB0(include_top=False, weights="imagenet",
                                 input_shape=(img_size, img_size, 3), pooling="avg")
    base_model.trainable = False

    inputs = layers.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    return model, base_model

efficientnetb0_sipakmed, efficientnetb0_base = build_efficientnetb0(len(sipakmed_classes), IMG_SIZE)
efficientnetb0_sipakmed.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
efficientnetb0_sipakmed.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │         6,405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## Train on SIPaKMeD

Early stopping watches validation loss so it doesn't keep training once it stops improving, and the checkpoint keeps the best version seen, not just whatever the last epoch happened to be.

In [6]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("/kaggle/working/efficientnetb0_sipakmed_best.keras",
                                        monitor="val_loss", save_best_only=True),
]

history = efficientnetb0_sipakmed.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
)

Epoch 1/30


2026-09-26 21:55:30.549993: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:55:30.693558: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:55:31.060546: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:55:31.202290: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:55:31.995947: E external/local_xla/xla/stream_

88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.5328 - loss: 1.2093

2026-09-26 21:55:59.707762: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:55:59.849927: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:00.196533: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:00.337636: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:01.156941: E external/local_xla/xla/stream_

89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - accuracy: 0.5342 - loss: 1.2063

2026-09-26 21:56:22.225697: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:22.366734: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:22.709589: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:22.851073: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:56:23.643271: E external/local_xla/xla/stream_

89/89 ━━━━━━━━━━━━━━━━━━━━ 76s 558ms/step - accuracy: 0.6627 - loss: 0.9455 - val_accuracy: 0.8278 - val_loss: 0.5518
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.8157 - loss: 0.5554 - val_accuracy: 0.8930 - val_loss: 0.4199
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.8470 - loss: 0.4670 - val_accuracy: 0.8997 - val_loss: 0.3597
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.8575 - loss: 0.4074 - val_accuracy: 0.9130 - val_loss: 0.3278
Epoch 5/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.8737 - loss: 0.3741 - val_accuracy: 0.9147 - val_loss: 0.3023
Epoch 6/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.8874 - loss: 0.3428 - val_accuracy: 0.9130 - val_loss: 0.2888
Epoch 7/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.8794 - loss: 0.3411 - val_accuracy: 0.9164 - val_loss: 0.2821
Epoch 8/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.8962 - loss: 0.3162 - val_accuracy: 0.9281 - val_loss: 

## Evaluate and save the checkpoint

This checkpoint is what CRIC's and Mendeley's severity fine-tuning stages will start from next, not raw ImageNet weights, so saving it here matters.

In [7]:
test_loss, test_acc = efficientnetb0_sipakmed.evaluate(test_ds)
print(f"SIPaKMeD test accuracy: {test_acc:.3f}")

efficientnetb0_sipakmed.save("/kaggle/working/efficientnetb0_sipakmed_final.keras")
print("Saved. This becomes the starting checkpoint for the CRIC and Mendeley fine-tuning stages next.")

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 129ms/step - accuracy: 0.9227 - loss: 0.2366
SIPaKMeD test accuracy: 0.923
Saved. This becomes the starting checkpoint for the CRIC and Mendeley fine-tuning stages next.


## Build ConvNeXt with a new head for SIPaKMeD's 5 classes

Same setup as EfficientNetB0: ImageNet-pretrained, base frozen, new head trained from scratch on SIPaKMeD. ConvNeXtTiny is the right size here, it's the smallest variant and keeps this in the same ballpark as EfficientNetB0 for a fair comparison later, and for GPU budget.

In [8]:
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess_input

def build_convnext(num_classes, img_size):
    base_model = ConvNeXtTiny(include_top=False, weights="imagenet",
                               input_shape=(img_size, img_size, 3), pooling="avg")
    base_model.trainable = False

    inputs = layers.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    return model, base_model

convnext_sipakmed, convnext_base = build_convnext(len(sipakmed_classes), IMG_SIZE)
convnext_sipakmed.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
convnext_sipakmed.summary()

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 768)            │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         3,845 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,823,973 (106.14 MB)

 Trainable params: 3,845 (15.02 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

## Rebuild the data pipeline using ConvNeXt's own preprocessing

Each architecture normalizes pixel values slightly differently, so the datasets need rebuilding with ConvNeXt's `preprocess_input` instead of EfficientNet's, even though everything else about the pipeline is identical.

In [9]:
def make_dataset_convnext(df, class_to_idx, img_size, batch_size, shuffle=False, augment=False):
    filepaths = df["filepath"].values
    labels = df["label"].map(class_to_idx).values

    def load_and_preprocess(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [img_size, img_size])
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
            image = tf.image.random_brightness(image, 0.1)
        image = convnext_preprocess_input(image)
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds_convnext = make_dataset_convnext(sipakmed_train_df, class_to_idx, IMG_SIZE, BATCH_SIZE, shuffle=True, augment=True)
val_ds_convnext = make_dataset_convnext(sipakmed_val_df, class_to_idx, IMG_SIZE, BATCH_SIZE)
test_ds_convnext = make_dataset_convnext(sipakmed_test_df, class_to_idx, IMG_SIZE, BATCH_SIZE)

## Train ConvNeXt on SIPaKMeD

In [10]:
callbacks_convnext = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("/kaggle/working/convnext_sipakmed_best.keras",
                                        monitor="val_loss", save_best_only=True),
]

history_convnext = convnext_sipakmed.fit(
    train_ds_convnext,
    validation_data=val_ds_convnext,
    epochs=30,
    callbacks=callbacks_convnext,
)

Epoch 1/30


2026-09-26 21:58:03.583026: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:58:03.718675: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.4331 - loss: 1.5624

2026-09-26 21:58:21.862660: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:58:21.997826: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 206ms/step - accuracy: 0.4347 - loss: 1.5576

2026-09-26 21:58:34.035879: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-26 21:58:34.171447: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


89/89 ━━━━━━━━━━━━━━━━━━━━ 45s 348ms/step - accuracy: 0.5790 - loss: 1.1374 - val_accuracy: 0.8227 - val_loss: 0.5303
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 16s 175ms/step - accuracy: 0.8252 - loss: 0.5067 - val_accuracy: 0.8746 - val_loss: 0.3984
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 173ms/step - accuracy: 0.8572 - loss: 0.4080 - val_accuracy: 0.8729 - val_loss: 0.3449
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 168ms/step - accuracy: 0.8797 - loss: 0.3422 - val_accuracy: 0.8813 - val_loss: 0.3215
Epoch 5/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 166ms/step - accuracy: 0.8959 - loss: 0.3041 - val_accuracy: 0.8896 - val_loss: 0.2980
Epoch 6/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 166ms/step - accuracy: 0.9033 - loss: 0.2791 - val_accuracy: 0.8997 - val_loss: 0.2887
Epoch 7/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 168ms/step - accuracy: 0.9089 - loss: 0.2645 - val_accuracy: 0.9064 - val_loss: 0.2676
Epoch 8/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 15s 169ms/step - accuracy: 0.9075 - loss: 0.2545 - val_accuracy: 0.904

## Evaluate and save

In [11]:
test_loss_convnext, test_acc_convnext = convnext_sipakmed.evaluate(test_ds_convnext)
print(f"ConvNeXt SIPaKMeD test accuracy: {test_acc_convnext:.3f}")

convnext_sipakmed.save("/kaggle/working/convnext_sipakmed_final.keras")
print("Saved. Same next steps as EfficientNetB0: Save Version, then New Dataset from the Output tab.")

19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.9293 - loss: 0.2337
ConvNeXt SIPaKMeD test accuracy: 0.929
Saved. Same next steps as EfficientNetB0: Save Version, then New Dataset from the Output tab.


## RegNet via PyTorch: data pipeline

Since torchvision has RegNet with ImageNet weights, this builds the same train/val/test split into a PyTorch dataset instead of a tf.data one. RegNetY_400MF is used, it's the size that keeps this comparable to EfficientNetB0 and ConvNeXtTiny rather than picking something drastically bigger.

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import regnet_y_400mf, RegNet_Y_400MF_Weights
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE_TORCH = 224
BATCH_SIZE_TORCH = 32

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_TORCH, IMG_SIZE_TORCH)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE_TORCH, IMG_SIZE_TORCH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

class SipakmedDataset(Dataset):
    def __init__(self, df, class_to_idx, transform):
        self.filepaths = df["filepath"].values
        self.labels = df["label"].map(class_to_idx).values
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        image = Image.open(self.filepaths[idx]).convert("RGB")
        image = self.transform(image)
        return image, int(self.labels[idx])

train_dataset_torch = SipakmedDataset(sipakmed_train_df, class_to_idx, train_transform)
val_dataset_torch = SipakmedDataset(sipakmed_val_df, class_to_idx, eval_transform)
test_dataset_torch = SipakmedDataset(sipakmed_test_df, class_to_idx, eval_transform)

train_loader = DataLoader(train_dataset_torch, batch_size=BATCH_SIZE_TORCH, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset_torch, batch_size=BATCH_SIZE_TORCH, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset_torch, batch_size=BATCH_SIZE_TORCH, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset_torch)}  Val: {len(val_dataset_torch)}  Test: {len(test_dataset_torch)}")

Using device: cuda
Train: 2843  Val: 598  Test: 608


## Build RegNet with a new head for SIPaKMeD's 5 classes

Same idea as the other two: pretrained on ImageNet, base frozen, only the new final layer trains.

In [13]:
regnet_model = regnet_y_400mf(weights=RegNet_Y_400MF_Weights.IMAGENET1K_V2)

for param in regnet_model.parameters():
    param.requires_grad = False

num_features = regnet_model.fc.in_features
regnet_model.fc = torch.nn.Linear(num_features, len(sipakmed_classes))
regnet_model = regnet_model.to(device)

optimizer = torch.optim.Adam(regnet_model.fc.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

print(regnet_model.fc)

Downloading: "https://download.pytorch.org/models/regnet_y_400mf-e6988f5f.pth" to /root/.cache/torch/hub/checkpoints/regnet_y_400mf-e6988f5f.pth


100%|██████████| 16.8M/16.8M [00:00<00:00, 118MB/s] 


Linear(in_features=440, out_features=5, bias=True)


## Train RegNet on SIPaKMeD

PyTorch doesn't have a built-in `.fit()`, so this is a manual loop doing the same job as the EarlyStopping/ModelCheckpoint callbacks did before: track validation loss each epoch, save whenever it improves, and stop after 5 epochs with no improvement.

In [14]:
import copy

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)

    return total_loss / total, correct / total

EPOCHS = 30
PATIENCE = 5
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(regnet_model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(regnet_model, val_loader, criterion)

    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
          f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(regnet_model.state_dict())
        epochs_without_improvement = 0
        torch.save(best_state, "/kaggle/working/regnet_sipakmed_best.pt")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}, restoring best weights.")
            break

regnet_model.load_state_dict(best_state)
print("Restored best weights (lowest val_loss).")

Epoch 1/30 - loss: 1.0640 - accuracy: 0.6349 - val_loss: 0.7042 - val_accuracy: 0.7943
Epoch 2/30 - loss: 0.6276 - accuracy: 0.8055 - val_loss: 0.5156 - val_accuracy: 0.8495
Epoch 3/30 - loss: 0.5019 - accuracy: 0.8417 - val_loss: 0.4441 - val_accuracy: 0.8528
Epoch 4/30 - loss: 0.4584 - accuracy: 0.8456 - val_loss: 0.3996 - val_accuracy: 0.8662
Epoch 5/30 - loss: 0.4222 - accuracy: 0.8579 - val_loss: 0.3782 - val_accuracy: 0.8712
Epoch 6/30 - loss: 0.3918 - accuracy: 0.8677 - val_loss: 0.3531 - val_accuracy: 0.8963
Epoch 7/30 - loss: 0.3929 - accuracy: 0.8656 - val_loss: 0.3416 - val_accuracy: 0.8846
Epoch 8/30 - loss: 0.3622 - accuracy: 0.8762 - val_loss: 0.3411 - val_accuracy: 0.8829
Epoch 9/30 - loss: 0.3635 - accuracy: 0.8737 - val_loss: 0.3177 - val_accuracy: 0.8896
Epoch 10/30 - loss: 0.3614 - accuracy: 0.8737 - val_loss: 0.3002 - val_accuracy: 0.9080
Epoch 11/30 - loss: 0.3334 - accuracy: 0.8829 - val_loss: 0.3181 - val_accuracy: 0.9013
Epoch 12/30 - loss: 0.3348 - accuracy: 0.

## Evaluate and save

In [15]:
test_loss, test_acc = run_epoch(regnet_model, test_loader, criterion)
print(f"RegNet SIPaKMeD test accuracy: {test_acc:.3f}")

torch.save(regnet_model.state_dict(), "/kaggle/working/regnet_sipakmed_final.pt")
print("Saved. Same next steps: Save Version, then New Dataset from the Output tab.")

RegNet SIPaKMeD test accuracy: 0.913
Saved. Same next steps: Save Version, then New Dataset from the Output tab.
